# Load audio and pick peaks from one DFT

A short viola excerpt is trimmed, faded, and analysed with a spectrogram, a mel
spectrogram, and a single-frame spectrum with peak picking. The source is the
CCRMA viola MP3. Read `waveforms_sampling_dft` first if the spectrum plot is new.

In [ ]:
from pathlib import Path
import audiospylt as asp

## Load

`load_audio` accepts a path or an http(s) URL. An MP3 is not converted to a WAV
file: libsndfile decodes it straight into a float64 NumPy array, as it does for
a WAV, and the array keeps the file's sample rate. MP3 is lossy, so the samples
carry its compression artefacts: the analysis below sees the decoded signal, not
the original recording. With no `download_dir`, a URL is decoded in memory and
nothing is written next to the notebook. Channels are averaged to mono by
default; this file has one.

In [ ]:
url = "https://ccrma.stanford.edu/~jos/mp3/viola.mp3"
audio, sr = asp.load_audio(url)
print(f"sr={sr} Hz, {len(audio)} samples, {len(audio) / sr:.3f} s")
asp.nb.show(asp.viz.plot_waveform(audio, sr))
asp.nb.play(audio, sr)

## Trim and fade

Keep 4.37–4.75 s and apply a 0.03 s fade-in and a 0.05 s fade-out. The fades
smooth the cut edges of the excerpt. Listen for two bowed notes: C4, then D4.

In [ ]:
start, stop, fade_in, fade_out = 4.37, 4.75, 0.03, 0.05
excerpt = asp.fade(asp.trim(audio, sr, start, stop), sr, fade_in=fade_in, fade_out=fade_out)
print(f"excerpt {len(excerpt)} samples, {stop - start:.3f} s")
asp.nb.show(asp.viz.plot_waveform(excerpt, sr))
asp.nb.play(excerpt, sr)

## Spectrogram

`spectrogram` builds a short-time transform. Passing `audio=excerpt` draws the
waveform above the time–frequency image. `freq_scale="log"` gives every octave
the same height, so a whole tone is the same step in each partial, and `fmin` /
`fmax` crop the view to 100–8000 Hz. Before you run the cell: where in the
excerpt does the pitch step from C4 to D4?

In [ ]:
sg = asp.spectrogram(excerpt, sr)
step_in, file_step = 0.19, start + 0.19
print(f"frames={sg.times.size}, bins={sg.freqs.size}; step ~{step_in} s in (file {file_step:.2f} s)")
view = dict(freq_scale="log", fmin=100, fmax=8000)
asp.nb.show(asp.viz.plot_spectrogram(sg, audio=excerpt, **view))

The step sits near 0.19 s into the excerpt (file time 4.56 s): C4 near 262.7 Hz
then D4 near 292.3 Hz (rising).

## Frame length and hop

The same excerpt with a short frame (`frame_length=512`, `hop=128`) and a long
one (`frame_length=8192`, `hop=2048`). Bin spacing is `sr / frame_length` in Hz;
the frame step is `hop / sr` in seconds.

In [ ]:
for frame_length, hop in ((512, 128), (8192, 2048)):
    s = asp.spectrogram(excerpt, sr, frame_length=frame_length, hop=hop)
    print(f"N={frame_length}, hop={hop}: bin {sr / frame_length:.2f} Hz, "
          f"step {hop / sr:.4f} s, frame {frame_length / sr:.4f} s, {s.times.size} frames")
    asp.nb.show(asp.viz.plot_spectrogram(s, audio=excerpt, **view))

The short frame draws the C4 → D4 change sharper in time (0.0029 s steps, 86.13
Hz bins). The long frame separates the harmonics (5.38 Hz bins) but its frames
step 0.0464 s apart and each spans 0.1858 s (8192 samples), so the change is
smeared.

## Mel spectrogram

`mel_spectrogram` takes a `Spectrogram` and folds it onto mel bands. Here
`n_mels=64` and `fmax=8000` set the band count and the top of the filterbank
(defaults are 128 bands up to Nyquist). It needs the `librosa` extra
(`pip install "audiospylt[librosa]"`).

In [ ]:
mel = asp.mel_spectrogram(sg, n_mels=64, fmax=8000)
asp.nb.show(asp.viz.plot_spectrogram(mel))
print(f"n_mels={mel.n_mels}, fmax={mel.fmax:g} Hz, frames={mel.times.size}")

## Spectrum and peaks

`spectrum` analyses the whole excerpt as one frame, with a Hann window by default
(`window=` picks another). `plot_waveform(..., window=spec.window)` shows what the
transform receives: the excerpt after the window, with the window curve on the
right-hand axis. The window peaks at 0.19 s, close to the C4 to D4 step at about
0.20 s, so the change itself gets the most weight.

`min_db` is an absolute
floor in dB re full scale; `prominence_db` is how far a peak must rise above its
neighbours; `fmin` / `fmax` limit the band. `help(asp.find_peaks)` also lists
`rel_db` (a floor relative to the strongest peak). The choices below are v1
settings, not a numeric translation of the old linear thresholds.

Before you run the cell: with `prominence_db=12`, does a peak near 291 Hz (the
D4 fundamental) survive?

In [ ]:
spec = asp.spectrum(excerpt, sr)
peaks = asp.find_peaks(spec, min_db=-45, prominence_db=12, fmin=100, fmax=3500)
d4 = asp.find_peaks(spec, min_db=-45, prominence_db=10, fmin=280, fmax=300)
f0, strong = float(peaks.freq.iloc[0]), peaks.loc[peaks.amp.idxmax()]
h2 = float(peaks.loc[(peaks.freq > 560) & (peaks.freq < 610), "freq"].iloc[0])
print(f"{len(peaks)} peaks; strongest {strong.freq:.1f} Hz ({asp.amplitude_to_db(strong.amp):.1f} dB)")
print(f"D4 at prom>=10: {d4.freq.iloc[0]:.1f} Hz, prominence {d4.prominence.iloc[0]:.1f} dB")
print(f"D4 series f0 ~ {h2 / 2:.1f} Hz from partial {h2:.1f} Hz")
print(peaks.assign(ratio=peaks.freq / f0)[["freq", "amp", "ratio"]].round(3).to_string())
asp.nb.show(asp.viz.plot_waveform(excerpt, sr, window=spec.window))
asp.nb.show(asp.viz.plot_spectrum(spec, peaks))

Thirteen peaks remain between 100 Hz and 3500 Hz. Rows are sorted by frequency,
not amplitude; the strongest peak is 262.7 Hz (−21.6 dB), about C4. The D4
fundamental at 291.1 Hz has prominence 10.4 dB, so `prominence_db=12` drops it
(the glide smears the peak). What remains is two harmonic series: ratios near
1, 2, 3, 4, 8 × 262.7 Hz (C4) and near 2, 3, 4, 7 × 292.3 Hz (D4 without its
fundamental).

## Write the peak table

`write_table` adds the `# audiospylt-table: snapshot v1` header. Create `tsv/`
in the writing cell.

In [ ]:
Path("tsv").mkdir(exist_ok=True)
path = asp.write_table(peaks, "tsv/viola_excerpt.tsv")
print(path)
print(path.read_text().splitlines()[0])